# Introduction

In this tutorial, we will use the **UNI model** from [MahmoodLab](https://github.com/mahmoodlab/UNI) (follow the Installation steps), a self-supervised vision transformer trained on histology slides, to extract **patch-level feature embeddings** from Whole Slide Images (WSIs).

The steps we will follow are:

1. **Load** the pretrained UNI model and its checkpoint weights.
2. **Tokenize** the WSI into small, non-overlapping patches of size **224×224 pixels**.
3. **Save** each patch image and record its spatial location (X, Y coordinates) on the original WSI.
4. **Feed** the patches through the pretrained UNI model to extract deep **feature embeddings**.
5. **Link** each embedding with its spatial metadata (Slide ID, Patch ID, coordinates).
6. **Save** the final dataset for downstream spatial analysis tasks.

---

By following this process, we create a compact and meaningful representation of very large WSIs, which can be easily used for downstream tasks.


# Imports and Environment Setup

In this section, we import the necessary libraries and set up the environment.

- **PyTorch** (`torch`) and **TorchVision**: for deep learning operations and image handling.
- **Pandas**, **NumPy**: for working with tabular data and arrays.
- **PIL** (`Image`): for opening and processing images.
- **Tifffile**: for reading `.tiff` format images, commonly used in digital pathology.
- **UNI model utilities**: 
  - `get_encoder` to load the pretrained UNI model.
  - `extract_patch_features_from_dataloader` to extract embeddings from patches.
  - Various evaluation utilities for downstream analysis (e.g., linear probe, k-NN, ProtoNet).

Finally, we set the **device** to use a **GPU** if available, otherwise fallback to **CPU**:
```python
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [1]:
import torch
import torchvision
import sys
import os
from os.path import join as j_
from PIL import Image
import pandas as pd
import numpy as np
import tifffile
import numpy as np
import os
from PIL import Image
import anndata as ad

sys.path.append('..')
from uni import get_encoder
from uni.downstream.extract_patch_features import extract_patch_features_from_dataloader
from uni.downstream.eval_patch_features.linear_probe import eval_linear_probe
from uni.downstream.eval_patch_features.fewshot import eval_knn, eval_fewshot
from uni.downstream.eval_patch_features.protonet import ProtoNet, prototype_topk_vote
from uni.downstream.eval_patch_features.metrics import get_eval_metrics, print_metrics
from uni.downstream.utils import concat_images

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

/home/charlottewolf/miniconda3/envs/uni/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Loading the Pretrained UNI Model and Preparing Image Transformations

In this section, we load the **pretrained UNI model** and prepare the **transformations** that will be applied to the image patches before feeding them into the model.

Steps:

1. **Import additional libraries**:
   - `timm`: a popular library that provides implementations of vision transformer models.
   - `huggingface_hub`: for downloading model weights (optional here if already downloaded).

2. **Set the local directory**:
   - `local_dir` is where the pretrained model checkpoint (`pytorch_model.bin`) is stored. Here 'checkpoint_path' is an exampl, set where your checkpoint is located.
   - (Optional) If not downloaded yet, you could use `hf_hub_download` to fetch the weights from Hugging Face.

3. **Model setup**:
   - We define the model architecture using `timm_kwargs` to match the configuration used by UNI (based on a Vision Transformer (ViT) backbone).
   - The model is created with `timm.create_model(**timm_kwargs)`.
   - Pretrained weights are loaded with `model.load_state_dict(...)`.
   - The model is set to evaluation mode (`model.eval()`) and moved to the selected device (CPU or GPU).

4. **Image preprocessing**:
   - Before passing patches to the model, we apply a series of transformations:
     - **Resize** the patch to 224×224 pixels.
     - **Convert** the image to a PyTorch tensor.
     - **Normalize** pixel values using ImageNet statistics (mean and std).



In [2]:
import os
import torch
from torchvision import transforms
import timm
#from huggingface_hub import login, hf_hub_download

#login(token="xxx")  
pretrained = True

local_dir = '../weights/'
os.makedirs(local_dir, exist_ok=True)  
#hf_hub_download("MahmoodLab/UNI2-h", filename="pytorch_model.bin", local_dir=local_dir, force_download=True)
timm_kwargs = {'model_name': 'vit_giant_patch14_224',
    'img_size': 224, 
               'patch_size': 14, 
               'depth': 24,
               'num_heads': 24,
               'init_values': 1e-5, 
               'embed_dim': 1536,
               'mlp_ratio': 2.66667*2,
               'num_classes': 0, 
               'no_embed_class': True,
               'mlp_layer': timm.layers.SwiGLUPacked, 
               'act_layer': torch.nn.SiLU, 
               'reg_tokens': 8, 
               'dynamic_img_size': True
              }
model = timm.create_model(**timm_kwargs)
model.load_state_dict(torch.load(os.path.join(local_dir, "pytorch_model.bin"), map_location="cpu"), strict=True)
model.eval()
model.to(device)
transform = transforms.Compose(
    [
        transforms.Resize(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ]
)


/tmp/ipykernel_6982/584555659.py:29: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(os.path.join(local_dir, "pytorch_model.bin"), map_locatio

# Tokenizing Whole Slide Images (WSIs) into Patches

In this section, we split each Whole Slide Image (WSI) into small non-overlapping patches and save them as individual image files, along with recording their spatial metadata.

Steps:

1. **Set up paths**:
   - `person`: data is differntiated through person IDs. From 10x: Colorectal Cancer: CRC_P1, CRC_P2, CRC_P5. Non-cancerous: Normal_P3, Normal_P5.
   - `bin_size`: Visium HD bin size. Choose between 002, 008, 016 µm. 
   - `filename`: Filename os the input image
   - `filepath`: Filepath of the input image. Note that you *do not have to edit this*, if you follow the folder structure given in the .git instructions. 
   - `output_base_folder`: where extracted patches will be saved. (set your location)
   - `metadata_file`: CSV file to store patch metadata (Slide_ID, Patch_ID, X, Y, file path). (set your file name and location or keep it as the default)

2. **Patch parameters**:
   - **Patch size**: `224 × 224` pixels.
   - **Stride**: `224` pixels (non-overlapping patches).

3. **Retreiving the frame coordinates**:
   - 10x genomics/the spaceranger pipeline provide a `tissuepositions.parquet` file, which notes the pixel postion for each bin. 
   From here, we extract the top left and bottom right pixel corner of the image. We use this later to crop only the necessary part from the `.btf`-file, since it is very large and captures the entire tissue slide, not just our desired frame. 

4. **Processing the WSI**:
   - Load the `.btf` file using `tifffile.imread()`.
   - Convert the numpy array to a PIL image for easier patch extraction.
   - Loop over the WSI grid and **crop** patches at regular intervals.
   - Save each patch as a `.png` image.
   - Record metadata for each patch:
     - Slide ID (origin WSI)
     - Patch filename
     - Top-left (X, Y) coordinate of the patch within the slide
     - Path where the patch is stored

5. **Handling errors**:
   - If a WSI fails to load or process, it is added to `failed_wsi_files`.

6. **Metadata saving**:
   - After processing all WSIs, the collected metadata is stored in a single CSV file (`WSI_patch_metadata_prostate.csv`).

---

The metadata will also be important for linking back embeddings to their original spatial locations on the WSI.


In the following, please choose the person (at the time of writing: either CRC_Px or Normal_Px) and the bin size (2um, 8um or 16um).

In [ ]:
person = "CRC_P5" # Adapt to requirements
bin_size = "002"  # Adapt to requirements

filename = "Visium_HD_Human_Colon_Cancer_P5_tissue_image.btf"  # Adapt to requirements
filepath = f"../../data/{person}/input/{filename}"

if not os.path.exists("metadata"):
    os.makedirs("metadata")

metadata_csv_file = "metadata/WSI_patch_metadata.csv"   
metadata_h5ad_file = "metadata/WSI_patch_metadata.h5ad"  

output_base_folder = "patches" #Where you want to save the patches
os.makedirs(output_base_folder, exist_ok=True)

patch_size = (224, 224)  
stride = 224
root_path = output_base_folder

Get the minimum pixel coordinates for the top left pixel corner and the maximum pixel coordinates for the bottom right corner of the desired frame/region to crop. 

In [4]:
# Get crop cordinates for spatial transcirptomics area from the "tissue_positions.parquet" file
import pandas as pd
import numpy as np
import math
from PIL import Image

col = "pxl_col_in_fullres"
row = "pxl_row_in_fullres"  

# Get crop cordinates for spatial transcirptomics area from the "tissue_positions.parquet" file
tissuepositions = pd.read_parquet(f"../../data/{person}/output/binned_outputs/square_{bin_size}um/spatial/tissue_positions.parquet")

min_coordinates = [tissuepositions.loc[tissuepositions[col] > 0, col].min(), tissuepositions.loc[tissuepositions[row] > 0, row].min()]
max_coordinates = [tissuepositions.loc[tissuepositions[col] > 0, col].max(), tissuepositions.loc[tissuepositions[row] > 0, row].max()]
print(f"Minimum coordinates: {min_coordinates} (row, column)")
print(f"Maximum coordinates: {max_coordinates} (row, column)")

# Coordinates for cropping (top-left corner)
#x0, y0 = math.floor(min_coordinates[0]), math.floor(min_coordinates[1])
#width, height = math.ceil(max_coordinates[0]) - x0, math.ceil(max_coordinates[1]) - y0
xmin, ymin = math.floor(min_coordinates[0]), math.floor(min_coordinates[1])
xmax, ymax = math.ceil(max_coordinates[0]), math.ceil(max_coordinates[1])

Minimum coordinates: [44268.96012379498, 39511.75092759462] (row, column)
Maximum coordinates: [68834.38155427143, 64076.159171259045] (row, column)


1. Load the image partially
2. Crop out desired region
3. Optionally save to .tif file
4. Convert to numpy array

In [10]:
import pyvips

# Open the image (does not load it fully)
image = pyvips.Image.new_from_file(filepath, access="sequential")

cropped = image.crop(xmin, ymin, xmax - xmin, ymax - ymin)

# Convert to NumPy or Save to disk or (if needed)

# Uncomment to save the cropped region to a file:
# wsi_image.write_to_file("cropped_region.tif")

# Convert to NumPy array
numpy_array = np.ndarray(
    buffer=cropped.write_to_memory(),
    dtype=np.uint8,
    shape=[cropped.height, cropped.width, cropped.bands],
)

# Convert to PIL image
wsi_image = Image.fromarray(numpy_array)

In [11]:
failed_wsi_files = []
all_patch_metadata = [] 

try:
    slide_name = person # Use the file name without extension and add person identifier
    slide_output_folder = os.path.join(output_base_folder, slide_name)
    os.makedirs(slide_output_folder, exist_ok=True)

    wsi_width = xmax - xmin
    wsi_height = ymax - ymin

    print(f"Processing {slide_name} - Size: {wsi_width}x{wsi_height}")

    saved_patches = 0
    for x in range(0, wsi_width - patch_size[0], stride):
        for y in range(0, wsi_height - patch_size[1], stride):
            patch = wsi_image.crop((x, y, x + patch_size[0], y + patch_size[1]))
        
            patch_filename = f"patch_{saved_patches+1}.png"
            patch_path = os.path.join(slide_output_folder, patch_filename)
            patch.save(patch_path)

            
            all_patch_metadata.append([slide_name, patch_filename, x, y, patch_path])
            saved_patches += 1

    print(f"Saved {saved_patches} patches for {slide_name}")

except Exception as e:
    print(f"Error processing {filename}: {e}")
    failed_wsi_files.append(filename)


metadata_df = pd.DataFrame(all_patch_metadata, columns=['Slide_ID', 'Patch_ID', 'X', 'Y', 'Path'])

if not os.path.exists(metadata_csv_file):
    metadata_df.to_csv(metadata_csv_file, index=False)
else:
    existing_metadata_csv = pd.read_csv(metadata_csv_file)
    metadata_df = pd.concat([existing_metadata_csv, metadata_df], ignore_index=True)
    metadata_df.to_csv(metadata_csv_file, index=False)

print("All WSIs processed successfully!")
print("Failed WSIs:", failed_wsi_files)
print(f"Metadata saved in {metadata_csv_file}")


Processing CRC_P5 - Size: 24567x24566
Saved 11881 patches for CRC_P5
All WSIs processed successfully!
Failed WSIs: []
Metadata saved in WSI_patch_metadata.csv


# Extracting Patch Feature Embeddings Using the UNI Model

In this section, we load the extracted patches, pass them through the pretrained UNI model, and save their feature embeddings along with spatial metadata.

Steps:

1. **Patch Dataset Preparation**:
   - Define a `PatchDataset` class that:
     - Loads all `.png` patch images from a given folder.
     - Applies the same image transformations we defined earlier.
     - Returns an image and a dummy label (not used here).

2. **Loading Patches and Creating DataLoader**:
   - For each WSI (each folder of patches):
     - Create a `PatchDataset` and corresponding `DataLoader`.
     - Load patches in small batches (batch size = 4) without shuffling (preserve patch order).

3. **Extract Patch Features**:
   - Use the `extract_patch_features_from_dataloader` function from UNI to pass the patches through the model.
   - Extract the feature embeddings for each patch.

4. **Collect Embeddings and Metadata**:
   - For each patch, store:
     - Its extracted embedding.
     - The corresponding Slide_ID (WSI name).
     - The Patch_ID (patch filename).

5. **Merge with Patch Location Metadata**:
   - Combine the extracted embeddings with the previously saved patch spatial information (`X`, `Y` coordinates) using a `match_id` (Slide_ID + Patch_ID).

6. **Save Final Data**:
   - Store everything in an Anndata object file `WSI_patch_embeddings_prostate.h5ad`, which contains:  
     AnnData object with n_obs × n_vars = n_patches × embedding_dim  
      ├── `.X`               → patch embeddings (from UNI model)  
      ├── `.obs`             → Slide_ID, Patch_ID, X, Y  
      ├── `.obsm["spatial"]` → [X, Y] coordinates for spatial plotting  
      └── `.uns`             → currently empty 
   - Optinal: save as .csv (currently out-commmented)

---

At the end of this step, we will have a **structured table** linking each patch's location to its **deep feature representation**, ready for downstream spatial analysis tasks.


In [ ]:
from uni.downstream.extract_patch_features import extract_patch_features_from_dataloader
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
import glob
from PIL import Image
import os
import pandas as pd
import torch

class PatchDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.image_paths = glob.glob(os.path.join(root_dir, "*.png"))  # Load all PNGs
        self.transform = transform
        self.patch_ids = [os.path.basename(img) for img in self.image_paths]  # Store Patch_IDs

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")  # Ensure 3 channels

        if self.transform:
            image = self.transform(image)
            
        return image, -1  


root_path = output_base_folder

valid_folders = [f for f in sorted(os.listdir(root_path)) if os.path.isdir(os.path.join(root_path, f)) and not f.startswith(".")]
print(f"Valid folders found: {valid_folders}")

all_embeddings = []
all_sample_ids = []
all_patch_ids = []  

for folder in valid_folders:
    folder_path = os.path.join(root_path, folder)

    test_dataset = PatchDataset(root_dir=folder_path, transform=transform)
    test_dataloader = DataLoader(test_dataset, batch_size=4, shuffle=False)

   
    test_features = extract_patch_features_from_dataloader(model, test_dataloader)
    test_feats = torch.Tensor(test_features['embeddings'])

    
    all_embeddings.append(test_feats.numpy())  # Convert to NumPy
    all_sample_ids.extend([folder] * test_feats.shape[0])  # Assign folder name to each patch
    all_patch_ids.extend(test_dataset.patch_ids)  

print(f"Length of all_embeddings: {len(all_embeddings)}")
for i, emb in enumerate(all_embeddings):
    print(f"Shape of all_embeddings[{i}]: {emb.shape}, type: {type(emb)}")


df_embeddings = pd.DataFrame(
    torch.cat([torch.tensor(arr) for arr in all_embeddings], dim=0).numpy()  # Flatten list of arrays
)
df_embeddings["Slide_ID"] = all_sample_ids 
df_embeddings["Patch_ID"] = all_patch_ids  

metadata_df = pd.read_csv(metadata_csv_file)  # Load metadata file
if metadata_df.empty:
    print("Warning: Metadata file is empty. Ensure it contains valid data.")

df_embeddings['match_id']=df_embeddings['Slide_ID']+'_'+df_embeddings['Patch_ID']
metadata_df['match_id']=metadata_df['Slide_ID']+'_'+metadata_df['Patch_ID']
df_embeddings=df_embeddings.merge(metadata_df[['match_id','X','Y']],on='match_id')
#df_embeddings.to_csv('test.csv',index=False) # uncomment to save intermediate results as csv if desired

metadata_cols = ["Slide_ID", "Patch_ID", "X", "Y"]
feature_cols = [col for col in df_embeddings.columns if col not in metadata_cols + ["match_id"]]

# Create AnnData object
adata = ad.AnnData(
    X=df_embeddings[feature_cols].values,      # shape (n_patches, embedding_dim)
    obs=df_embeddings[metadata_cols].copy()    # shape (n_patches, n_metadata_fields)
)

# Add spatial coordinates to .obsm for compatibility with spatial plotting
adata.obsm["spatial"] = df_embeddings[["X", "Y"]].values

# Optional: store Patch_ID as index
adata.obs.index = df_embeddings["Slide_ID"] + "_" + df_embeddings["Patch_ID"]

# Save to file
if os.path.exists(metadata_h5ad_file):
    existing_adata = ad.read_h5ad(metadata_h5ad_file)
    adata = ad.concat([existing_adata, adata], join='outer', label='dataset', index_unique=None)

adata.write("metadata/WSI_patch_embeddings.h5ad")

Valid folders found: ['CRC_P1', 'CRC_P2', 'CRC_P5']


100%|██████████| 2971/2971 [16:50<00:00,  2.94it/s]


Length of all_embeddings: 3
Shape of all_embeddings[0]: (11990, 1536), type: <class 'numpy.ndarray'>
Shape of all_embeddings[1]: (11110, 1536), type: <class 'numpy.ndarray'>
Shape of all_embeddings[2]: (11881, 1536), type: <class 'numpy.ndarray'>


/home/charlottewolf/miniconda3/envs/uni/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
